# 4팀 — LoL 10분 시점 승패 예측·이상탐지 API — 실습 노트북 (3시간 · MS1~MS4 + 체험)

MS1~MS4 에서 «남의 모델(감성 분류)» 로 한 일을 **자기 팀 모델** 로 한 번 더 한다. 새로 배우는 것은 없다.
이 노트북 하나로 **서버를 띄우고 · 부르고 · 깨뜨리고 · 재고 · 틀릴 때를 정한다.** 위에서 아래로 실행한다(Run All 가능).
MS5·MS6 는 따로 수업하지 않는다 — 마지막 «체험» 절에서 10분 안에 겪어 본다.

| 시간 | 절 | 하는 일 | 기록 |
|---|---|---|---|
| 0:00~0:15 | 0 준비 | 폴더 확인 · 파이썬 결정 · 서버 헬퍼 | — |
| 0:15~0:55 | 1 MS1 연결 | 서버 없이 `predict.py` → 팀 원본과 대조 → 서버 기동 → `/health` `/predict` | 기록 1 |
| 0:55~1:30 | 2 MS2 계약 | `schemas.py` 읽기 · `/docs` · 422 세 가지 만들기 · 한계 3줄 | 기록 2 |
| 1:30~2:05 | 3 MS3 측정 | `bench.py` 로 재기 · p95 로 약속 · 개선 과제 | 기록 3 |
| 2:05~2:45 | 4 MS4 실패 설계 | 판단보류 · 검토 큐 · 422/500 분리 · 정책 카드 | 기록 4 |
| 2:45~3:00 | 5 체험 | 버전 고정 · `.env` 로 정책 바꾸기 · 다섯 숫자 · pytest (MS5·MS6 를 대신한다) | 기록 5 |

**무엇을 하는 모델인가** — 10분 시점 경기 상태(블루−레드 차이 13개) → 블루 승리 확률·예측·승리요인 3개 + 이상탐지. 확률이 0.5±0.10 이면 «판단보류(접전)».

**이 폴더가 팀 원본에 더한 것** — 팀 정본은 루트 `artifacts/` + `lolwin/`(승패) 와 `models/anomaly_detect/`(이상탐지) 에 나뉘어 있었고 `models/win_predict/` 는 비어 있었다. `model/` 에 셋을 모으고, `predict.py` 가 한 입력으로 두 모델을 부른다.

> 팀 원본은 `model/` 에 그대로 있다. **`model/` 안은 고치지 않는다** — 고치는 순간 «팀 모델과 같은 값이 나온다» 는 증거(1절의 대조)가 사라진다.
> 팀마다 다른 파일은 `predict.py` `schemas.py` `settings.py` 셋뿐이고, `app.py` `metrics.py` `bench.py` 는 네 팀이 글자까지 같다.

## 0. 준비 — 폴더와 파이썬 (15분)

이 팀은 **python (이미지 그대로)** 로 실행한다. 필요한 패키지가 전부 이미지에 있다(모델을 만든 버전과 같다). 설치할 것이 없다.

노트북 커널은 이미지 파이썬이다. 모델을 부르는 일은 전부 **HTTP(서버)** 또는 **서브프로세스(`PY`)** 로 한다 — 커널에 팀 라이브러리를 설치하지 않아도 되는 이유다.

In [1]:
# 0-1 — 폴더 확인 · 파이썬 결정
import os, sys, json, time, subprocess, requests
from pathlib import Path

HERE = Path.cwd()
assert (HERE / "app.py").exists() and (HERE / "model").exists(), f"팀 폴더에서 열어야 한다: {HERE}"
NEED_VENV = False
PY = str(HERE / ".venv" / "bin" / "python") if NEED_VENV else sys.executable
PORT = 8000
BASE = f"http://127.0.0.1:{PORT}"
print("폴더:", HERE)
print("파이썬:", PY, "(venv 필요)" if NEED_VENV else "(이미지 그대로)")
print("파일:", sorted(p.name for p in HERE.iterdir() if not p.name.startswith(".")))

폴더: /home/jovyan/work/MSProject/team4_lol
파이썬: /opt/conda/bin/python (이미지 그대로)
파일: ['app.py', 'bench.py', 'examples', 'metrics.py', 'model', 'predict.py', 'requirements.txt', 'schemas.py', 'settings.py', 'team4_실습.ipynb', 'test_app.py']


In [2]:
# 0-끝 — 서버 띄우기/끄기 (오늘 여러 번 쓴다). 로그는 파일로 — 파이프는 가득 차면 서버가 멈춘다
def port_busy():
    try:
        return requests.get(f"{BASE}/health", timeout=1).ok
    except Exception:
        return False

def server_up():
    if port_busy():
        raise SystemExit(f"포트 {PORT} 에 이미 서버가 있다 — 다른 노트북의 서버라면 그쪽에서 server_down 후 다시")
    t0 = time.perf_counter()
    p = subprocess.Popen([PY, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", str(PORT)],
                         stdout=open("server.log", "a", encoding="utf-8"), stderr=subprocess.STDOUT, text=True, cwd=HERE)
    for _ in range(120):
        if port_busy():
            break
        if p.poll() is not None:
            raise SystemExit("서버가 죽었다 — server.log 마지막 줄을 본다:\n" + Path("server.log").read_text(encoding="utf-8")[-1500:])
        time.sleep(0.5)
    p.t_up = round(time.perf_counter() - t0, 2)
    print(f"서버 기동 {p.t_up}s · pid {p.pid}")
    return p

def server_down(p):
    p.terminate(); p.wait(timeout=10)
    print("서버 종료 · returncode", p.returncode)

def post(path, body, show=True):
    r = requests.post(f"{BASE}{path}", json=body, timeout=60)
    if show:
        print(r.status_code, json.dumps(r.json(), ensure_ascii=False)[:400])
    return r

REQ = json.loads(Path("examples/request.json").read_text(encoding="utf-8"))
BATCH = json.loads(Path("examples/request_batch.json").read_text(encoding="utf-8"))
print("예시 요청:", REQ)

예시 요청: {'FirstBlood': 1, 'KillsDiff': 5, 'GoldDiff': 4500, 'ExpDiff': 3000, 'WardsPlacedDiff': 5, 'WardsDestroyedDiff': 2, 'AssistsDiff': 6, 'DragonsDiff': 1, 'HeraldsDiff': 1, 'TowersDestroyedDiff': 1, 'AvgLevelDiff': 1.2, 'TotalMinionsKilledDiff': 30, 'TotalJungleMinionsKilledDiff': 10}


## 1. 연결 확인 (MS1 · 40분) — 서버 없이 먼저, 그다음 서버로

MS1 에서 한 순서 그대로다: `predict.py` 를 그냥 실행 → 서버 → `/health` → `/predict`.
서버가 안 뜨면 원인이 «모델» 인지 «서버» 인지 갈라야 하므로 **모델부터** 확인한다.

**팀 원본과 대조한다(서빙 파리티).** 감싼 코드가 팀 모델을 바꾸지 않았다는 증거다 — 팀 `predict.py --demo` — «블루가 크게 우세» **94.6%** · 골드 차이 기여 +2.242.

In [ ]:
# 1-1 — 서버 없이 predict.py (모델 적재 + 예시 몇 건). 각 필드를 schemas.py 의 PredictResponse 와 짝지어 읽는다
r = subprocess.run([PY, "predict.py"], capture_output=True, text=True, cwd=HERE)
print(r.stdout[-2500:] or r.stderr[-1500:])

In [ ]:
# 1-2 — 서버 기동 → /health → /predict (examples/request.json)
proc = server_up()
health = requests.get(f"{BASE}/health").json()
print("health:", health)
LOAD_SECONDS = health["load_seconds"]           # «첫 요청 지연» 의 재료 — 모델을 올린 시간
r = post("/predict", REQ)
FIRST = r.json()

### 기록 1 — 연결 (더블클릭해 채운다)

| 항목 | 값 (힌트) |
|---|---|
| 입력 → 출력 한 줄 (자기 말로) | 차이 13 → 블루 승리 확률·예측·요인 + 이상탐지 |
| `predict.py` 가 부르는 두 모델과 그 파일 | `model/lolwin` · `model/anomaly_detect` |
| 데모 요청의 블루 승리 확률 / 골드 차이 기여 | 94.6% · +2.242 |
| 이상탐지 결과 필드 이름과 값 |  |
| `load_seconds` (모델 적재 시간) |  |
| 이 서버의 값이 팀 원본 값과 일치하는가 (소수 넷째 자리) | 예 / 아니오 — 다르면 어느 층인가 |

> MS1 의 약속 세 줄이 여기서도 그대로다: 주소(`/predict`) · 입력 모양(`PredictRequest`) · 출력 모양(`PredictResponse`). `/docs` 에서 셋을 확인한다.

## 2. 계약 읽기와 깨뜨리기 (MS2 · 35분)

`schemas.py` 를 연다(왼쪽 파일 탐색기). 팀 원본의 스키마(팀 `model/artifacts/schema.json` 의 13피처·타입·학습 범위)와 나란히 놓고 **무엇을 옮겼고 무엇을 뺐는지** 찾는다.

422 를 세 가지 방법으로 만든다. 응답의 `detail` 을 읽고 **누가 막았는가**(pydantic 인가 · 팀 코드의 `ValueError` 인가)를 적는다. 둘 다 422 이지만 자리가 다르다 — `app.py` 의 `_run()` 이 `ValueError` 를 422 로 바꾼다.

In [ ]:
# 2-1 — /docs 주소 (JupyterHub 프록시 뒤에서는 아래 주소로 연다)
prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "").rstrip("/")
print(f"https://jupyterhub.sumzip.com{prefix}/proxy/{PORT}/docs" if prefix else f"{BASE}/docs")
print("Try it out 으로 examples/request.json 을 보내 본다")

In [ ]:
# 2-2 — 422 세 가지. 각각 detail 의 loc·msg 를 읽는다
CASES = [
    ("형식 위반 — 피처 누락(GoldDiff)", {k: v for k, v in REQ.items() if k != "GoldDiff"}),
    ("값 위반 — DragonsDiff 는 -1~1", {**REQ, **{"DragonsDiff": 3}}),
    ("타입 위반 — 문자열", {**REQ, **{"KillsDiff": "many"}}),
]
for label, body in CASES:
    r = requests.post(f"{BASE}/predict", json=body, timeout=30)
    d = r.json().get("detail")
    who = "팀 코드(ValueError)" if isinstance(d, str) else "pydantic(schemas.py)"
    print(f"{r.status_code}  {label}  ← {who}")
    print("     ", json.dumps(d, ensure_ascii=False)[:220])

In [ ]:
# 2-3 — 모르는 필드를 하나 더 넣는다 (extra="forbid"). 정답 컬럼 — 예측 입력이 아니다. `extra="forbid"` 가 막는다
post("/predict", {**REQ, "BlueWins": 1})

### 기록 2 — 계약

| 보낸 것 | 상태 | 누가 막았나 | `loc` | `msg` 한 줄 |
|---|---|---|---|---|
| 형식 위반 — 피처 누락(GoldDiff) | | | | |
| 값 위반 — DragonsDiff 는 -1~1 | | | | |
| 타입 위반 — 문자열 | | | | |
| 모르는 필드 `BlueWins` | | | | |

| 항목 | 값 (힌트) |
|---|---|
| 13개 피처 중 범위가 -1~1 인 것 (`schema.json`) |  |
| `BlueWins` 를 넣으면 422 인 이유 — «정답» 이 입력에 있으면 무엇이 잘못되나 |  |
| 피처 누락 · 범위 · 타입 — 세 오류의 `detail.type` 값 | missing · … · … |

**한계 3줄** — 팀 원본 문서(`model/`)와 오늘 본 422·대조 결과를 근거로 **자기 말로** 적는다. «무엇을 못 하는가 · 얼마나 틀리는가 · 어디까지 믿는가».

1.
2.
3.

## 3. 재기 (MS3 · 35분) — 같은 자로

`bench.py` 는 `examples/request.json` 을 보낸다. 팀이 달라도 방법은 같다: 단건 100회 → p50·p95, 배치 크기별, 동시 수별.
**평균은 나쁜 경험을 지운다** — 약속은 p95 로 한다. MS3 에서 «재고 나서 정한다» 고 한 습관을 자기 모델에 쓴다.

개선 과제 (개선 후 열은 이것을 한 뒤 **같은 셀** 로 다시 잰 값): `.env` 에 `INCLUDE_ANOMALY=false` 를 넣고 다시 잰다 — 두 모델 중 어느 쪽이 시간을 쓰는가. 코드는 고치지 않는다.

In [ ]:
# 3-1 — 측정 (bench.py 를 커널에서 import — requests 만 쓰므로 venv 와 무관)
import importlib, bench
importlib.reload(bench)
bench.BASE = BASE
bench.warmup()
M1 = bench.measure();               print("단건 100회 :", M1)
MB = [bench.measure_batch(b) for b in (1, 4, 8)]
for m in MB: print("배치", m)
MC = [bench.measure_concurrent(w) for w in (1, 5)]
for m in MC: print("동시", m)

### 기록 3 — 측정표

| 항목 | 개선 전 | 개선 후 |
|---|---|---|
| 첫 요청 지연 (load_seconds) | | |
| 단건 p50 / p95 (ms) | | |
| 처리량 (req/s) | | |
| 배치 8 건당 ms | | |
| 동시 5 p95 (ms) | | |

- 이 API 가 약속할 p95 (한 문장 · «○○ 요청의 95% 는 ○ms 안에»):
- 배치와 단건 중 건당 더 빠른 쪽과 그 이유:
- 개선 과제로 바꾼 것 (파일·함수) 과 결과:
- 우세(0.946) → 접전(0.512) → 열세(0.168) 순으로 확률이 내려가야 순서가 맞다 → 확인 [ ] 맞다 [ ] 아니다

## 4. 틀릴 때의 동작 (MS4 · 40분)

`app.py` 에서 세 곳을 찾는다: `ValueError → 422` · 그 밖 → `500` · `label == "판단보류" → review_queue.jsonl`. 감성 분류 때와 같은 구조다.
MS4 의 세 가지 실패 — **입력이 틀림(422) · 서버가 틀림(500) · 모델이 자신 없음(판단보류)** — 를 자기 모델에서 하나씩 만들어 본다.

팀 데모 «팽팽한 접전»(0.512) — |0.512−0.5| = 0.012 < 0.10 → 판단보류. 팀 모델에는 없던 정책을 MS4 방식으로 더한 것.

In [ ]:
# 4-1 — 판단보류를 실제로 만든다 → 검토 큐에 한 줄이 남는가
ABSTAIN_REQ = BATCH["items"][2]
d = post("/predict", ABSTAIN_REQ).json()
print("label:", d["label"], "· warnings:", d.get("warnings"))
print("검토 큐:", Path("review_queue.jsonl").read_text(encoding="utf-8").strip().splitlines()[-1] if Path("review_queue.jsonl").exists() else "(없음)")
# 큐에는 원문이 없다 — req_id 와 사유만. 원문은 로그에도 큐에도 넣지 않는다

In [ ]:
# 4-2 — 422 와 500 을 섞지 않는다: 지금까지의 요청을 metrics 로 본다 (error_rate 는 500 만 센다)
print(requests.get(f"{BASE}/metrics").json())
# 422 를 여러 번 냈는데 error_rate 가 0 인 이유를 한 줄로 적는다 → 기록 4

### 기록 4 — 정책 카드

임계값 «근거» 는 느낌이 아니라 **잰 것** 이어야 한다: 홀드아웃 1,976건 중 확률 0.4~0.6 구간의 정확도를 잰다(분할은 팀 프로젝트 `data/splits/`). 그 구간 정확도가 찍기(0.50)와 다르지 않다면 접전 폭 0.10 이 정당하다.

| 항목 | 값 (힌트) |
|---|---|
| 접전 판단보류 조건 (확률 0.5 ± ?) | 0.10 |
| 접전 폭 0.10 의 근거 — 홀드아웃 0.4~0.6 구간 정확도 (재서 적는다) |  |
| 우세 0.946 · 접전 0.512 · 열세 0.168 — 각각 label |  |

- 판단보류 한 건이 `review_queue.jsonl` 에 남긴 줄 (req_id · 사유):
- 더 비싼 오류: [ ] ________  [ ] ________ — 이유:
- 422 가 `error_rate` 에 안 잡히는 이유:
- 사람에게 넘길 때 함께 보낼 것 (MS4 3교시 — 원문은 넣지 않는다):

## 5. 체험 — 환경 고정과 운영 (15분 · MS5·MS6 를 대신한다)

MS5·MS6 는 수업으로 다루지 않는다. 대신 그 둘이 말하려는 것을 **이 폴더에서 세 번 겪어 본다.** 새 개념은 없다.

**(1) 버전을 적어 둔다 — `requirements.txt`.** 모델 파일은 «그것을 만든 라이브러리 버전» 에 묶여 있다. 같은 코드라도 라이브러리가 다르면 다른 답을 내거나 아예 열리지 않는다. 4팀 모델은 이미지의 scikit-learn 1.9.0 으로 만들어져 설치할 것이 없었다. 그래서 모든 줄에 `==` 로 버전을 박아 둔다 — 다음 사람이 같은 환경을 다시 만들 수 있게.

**(2) 정책값은 코드 밖에 둔다 — `.env`.** 4절의 임계값 같은 «정할 수 있는 값» 은 `settings.py` 가 `.env` 에서 읽는다. 값을 바꾸려고 코드를 고치면 테스트도 다시 해야 한다. `.env` 한 줄이면 **코드 무수정** 으로 서버가 다르게 행동한다. 접전 폭을 0.01 로 줄이면 0.512 는 «블루 승리 예측» 이 된다 — 코드는 안 고쳤다.

**(3) 서버가 사는 동안 다섯 숫자를 센다 — `/metrics`, 그리고 배포 전 3분 `pytest`.** 요청 수 · 오류율 · p50 · p95 · 마지막 요청 시각. 이 다섯이면 «지금 이 서버가 괜찮은가» 를 말할 수 있다. `test_app.py` 5종은 «이 다섯 가지 사고는 막았다» 는 증거다.

[주의] `.env` 에서 값 뒤에 같은 줄로 `#` 주석을 달면 주석이 값으로 읽힌다. 주석은 줄을 따로 쓴다.

In [ ]:
# 5-1 — (1) 버전 고정: requirements.txt 의 == 와 실행 파이썬의 실제 버전을 나란히
print(Path("requirements.txt").read_text(encoding="utf-8"))
r = subprocess.run([PY, "-c", "import sklearn, numpy, pandas; print('sklearn', sklearn.__version__, '· numpy', numpy.__version__, '· pandas', pandas.__version__)"],
                   capture_output=True, text=True, cwd=HERE)
print("실행 파이썬의 실제 버전:", r.stdout.strip() or r.stderr[-300:])

In [ ]:
# 5-2 — (2) .env 로 정책값을 바꾼다 (코드 무수정) → 서버 재기동 → 같은 요청이 다르게 답하는가
d0 = post("/predict", ABSTAIN_REQ, show=False).json(); print("바꾸기 전:", d0["label"])
server_down(proc)
Path(".env").write_text("MODEL_VERSION=team4-env-test\nCLOSE_MARGIN=0.01\n", encoding="utf-8")
proc = server_up()
print("health:", requests.get(f"{BASE}/health").json()["model_version"], "← .env 의 MODEL_VERSION")
d1 = post("/predict", ABSTAIN_REQ, show=False).json(); print("바꾼 뒤 :", d1["label"])
# 되돌린다 — .env 를 지우고 다시 띄운다 (제출물에 .env 는 없다 · .env.example 만)
server_down(proc); Path(".env").unlink(); proc = server_up()

In [ ]:
# 5-3 — (3) 다섯 숫자: 5-2 에서 서버를 다시 띄웠으므로 오늘 만든 요청을 한 번 더 보내고 센다 → 서버 종료 → 배포 전 3분(pytest 5종) → 로그
for _ in range(3): post("/predict", REQ, show=False)                 # 정상 3건
for _, body in CASES: requests.post(f"{BASE}/predict", json=body, timeout=30)   # 422 세 건 — error_rate 에 안 잡혀야 한다
post("/predict", ABSTAIN_REQ, show=False)                       # 판단보류 한 건
print("다섯 숫자:", json.dumps(requests.get(f"{BASE}/metrics").json(), ensure_ascii=False))
server_down(proc)                                   # 테스트는 TestClient 로 자기 앱을 띄운다
r = subprocess.run([PY, "-m", "pytest", "-q", "test_app.py"], capture_output=True, text=True, cwd=HERE)
print(r.stdout.strip().splitlines()[-1], "· 종료 코드", r.returncode, "(0 = 전부 통과)")
print("\n".join(Path("server.log").read_text(encoding="utf-8").strip().splitlines()[-10:]))

### 기록 5 — 체험

| 겪은 것 | 적을 것 | 값 |
|---|---|---|
| (1) 버전 고정 | 모델을 만든 라이브러리·버전 / `requirements.txt` 의 같은 줄 | |
| (2) 정책은 코드 밖 | `.env` 로 바꾼 키 → 같은 요청의 답이 어떻게 달라졌나 | |
| (3) 다섯 숫자 | requests · error_rate · p50 · p95 (5-3 출력) | |
| (3) 배포 전 3분 | pytest 결과 (5 passed?) · 막는 사고 하나를 자기 말로 | |

### 제출물 점검

| 파일 | 상태 |
|---|---|
| 이 노트북 | [ ] 기록 1~5 채움 |
| `test_app.py` | [ ] 5 passed (개선 과제로 `predict.py` 를 고쳤다면 그 뒤에도) |
| `server.log` 마지막 10줄 | [ ] 5-3 셀 출력 |

`.env` `.venv/` `*.jsonl` `*.log` 는 제출하지 않는다(`.gitignore`).

**막힐 때**: 포트 8000 사용 중 → `PORT = 8001` 로 바꾸고 0-끝 셀부터 · 서버가 이유 없이 죽으면 `predict.py` 첫머리의 `pd.set_option("future.infer_string", False)` 가 있는지 본다 — pandas 3 스레드 문제의 처방 · 500 인데 입력이 잘못됐다 → `server.log` 의 traceback 을 읽고 `predict.py` 에서 `ValueError` 로 바꾼다.